# ¿Por qué se equivocó el modelo?
**Estud-IA · Capa 2 · IA nivel intermedio · Encuentro 1**

Hoy **no entrenamos nada**. Usamos un modelo que ya existe, miramos qué predice y razonamos sobre sus aciertos y errores.

**Cómo se usa:** ejecuta cada celda en orden con el botón ▶ (o `Shift + Enter`). Al terminar cada paso escribe **LISTO** en el chat.

> Todos los mensajes de este notebook son **ficticios**.

## Paso 0 · Preparar el entorno
**Usa CPU, no GPU:** este notebook no necesita GPU. Si Colab te dice que no hay capacidad gratuita, ve a *Entorno de ejecución → Cambiar tipo de entorno de ejecución* y elige **CPU**.

Instala la biblioteca y descarga el modelo. La primera vez tarda 1 o 2 minutos.

In [ ]:
!pip -q install transformers sentencepiece

In [ ]:
from transformers import pipeline
import pandas as pd

NOMBRE_MODELO = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
clasificador = pipeline("text-classification", model=NOMBRE_MODELO, top_k=None)
print("Modelo cargado:", NOMBRE_MODELO)

## Paso 1 · Mira el modelo por dentro
Un modelo es **entrada → capas → pesos → salida**. Aquí contamos cuántas capas y cuántos pesos (parámetros) tiene.

In [ ]:
modelo = clasificador.model
tokenizador = clasificador.tokenizer

total_pesos = sum(p.numel() for p in modelo.parameters())
print("Capas del codificador:", modelo.config.num_hidden_layers)
print("Tamaño de cada representación:", modelo.config.hidden_size, "números")
print(f"Pesos (parámetros) en total: {total_pesos:,}")
print("Categorías de salida:", list(modelo.config.id2label.values()))

**Para comentar:** ¿te parece mucho o poco? Un modelo existente ya *aprendió* todos esos pesos. Adaptarlo es ajustar parte de ellos con **tus** datos (eso lo haremos más adelante).

## Paso 2 · Mensajes ficticios
Estos son mensajes que podría recibir el área digital de un museo ficticio. Abajo está la categoría que **nosotros** esperamos.

⚠️ **Todavía no ejecutes las predicciones.** Primero lee los mensajes y escribe en el chat qué categoría crees que va a elegir el modelo en cada uno: `negative`, `neutral` o `positive`.

In [ ]:
mensajes = [
    ("Las fotos de la exposición me encantaron, gracias por todo.",        "positive"),
    ("El producto llegó roto y nadie me responde.",                        "negative"),
    ("¿A qué hora abre el museo los domingos?",                            "neutral"),
    ("No estoy insatisfecho con el servicio.",                             "positive"),
    ("Qué chimba de visita guiada, repetiría sin dudarlo.",                "positive"),
    ("Esperé una hora en la fila y la guía ni siquiera sonrió.",           "negative"),
    ("La entrada costó 20.000 pesos.",                                     "neutral"),
    ("Pues sí, súper rápido el servicio... solo tardaron dos semanas.",    "negative"),
]
tabla = pd.DataFrame(mensajes, columns=["mensaje", "esperada"])
tabla

## Paso 3 · Ahora sí: mira las predicciones
Compara con lo que escribiste en el chat antes de ver el resultado.

In [ ]:
def predecir(texto):
    resultado = clasificador(texto)[0]
    mejor = max(resultado, key=lambda r: r["score"])
    return mejor["label"], round(mejor["score"], 2)

tabla[["prediccion", "confianza"]] = tabla["mensaje"].apply(lambda t: pd.Series(predecir(t)))
tabla["coincide"] = tabla["esperada"] == tabla["prediccion"]
tabla

In [ ]:
print(f"Coinciden {tabla['coincide'].sum()} de {len(tabla)} mensajes")
tabla[~tabla["coincide"]]

## Paso 4 · Compara un acierto y un error
Elige **un acierto** y **un error** de la tabla (cambia los números de abajo). Separa tres cosas:

1. **Preparación de la entrada:** ¿cómo se partió el texto en tokens?
2. **Representación:** ¿qué números recibe el modelo?
3. **Decisión del modelo:** ¿qué probabilidad le dio a cada categoría?

In [ ]:
INDICE = 3   # <- cambia este número para mirar otro mensaje (0 a 7)

texto = tabla.loc[INDICE, "mensaje"]
print("MENSAJE:", texto)
print("ESPERADA:", tabla.loc[INDICE, "esperada"])

print("\n1) Tokens:", tokenizador.tokenize(texto))
print("2) Números (ids):", tokenizador(texto)["input_ids"])

print("\n3) Probabilidad por categoría:")
for r in sorted(clasificador(texto)[0], key=lambda r: -r["score"]):
    print(f"   {r['label']:<9} {r['score']:.2f}")

### Preguntas para el grupo
- ¿Falla **la etiqueta** (¿estamos seguros de que nuestra categoría esperada es la correcta?), **el contexto** (ironía, jerga, negación) o **el modelo**?
- ¿Qué **dato adicional** ayudaría a saber cuál de las tres es?
- ¿Qué cambiarías en la **preparación de la entrada** para ver si el resultado cambia?
- Una persona: **reconstruye el razonamiento con tus palabras**.

> Un error no es un fracaso: es la pista de qué hay que comprobar después.

## Para llevarte
Escribe en el chat, en una línea: *¿en qué tarea tuya podría fallar un modelo así, y qué error sería el más grave?* Eso es la base de tu ficha de oportunidad.